# Synthetic label volumes with region analysis and overlap detection

Build two labeled 3D volumes, extract region properties with `RegionAnalyzer`, filter regions, then compute pairwise distances and overlap metrics for coincidence detection.

In [ ]:
import numpy as np
import pandas as pd
import stackview

from vistiq.utils import ArrayIteratorConfig
from vistiq.constant.matrix import FULL, LOWER, UPPER, LOWER_ND, UPPER_ND, OFF_DIAGONAL
from vistiq.segment.analysis import RegionAnalyzer, RegionAnalyzerConfig, region_to_numpy, dataframe_to_numpy
from vistiq.segment.select import RegionFilter, RegionFilterConfig, MinFilterConfig, RangeFilterConfig
from vistiq.segment.select import ValueFilter, ValueFilterConfig, TopKFilter, TopKFilterConfig
from vistiq.analysis import DistanceCalculator, DistanceCalculatorConfig
from vistiq.analysis.matrix import MatrixAggregator, MatrixAggregatorConfig
from vistiq.analysis.overlap import (
    LabelIntersectionCalculatorConfig,
    LabelOverlapCalculatorConfig,
    OverlapCalculator,
    metrics_calculator_configs,
    region_map_from_dataframe,
)

## Synthetic label volume

Shape `(100, 100, 10)` with labels `1`, `2`, and `3` in separate spatial regions (no overlap), mimicking a 3D segmentation mask.

In [ ]:
labels = np.zeros((10, 200, 200), dtype=np.uint64)

# Object 1 — upper-left
labels[2:7, 20:38, 12:30] = 1

# Object 2 — center
labels[3:9, 40:70, 38:62] = 2

# Object 3 — lower-right
labels[1:6, 72:92, 68:88] = 3

# Object 4 — lower-right
labels[4:6, 112:132, 125:163] = 4

# Object 5 — lower-right
labels[7:9, 145:180, 12:88] = 5

unique_labels = np.unique(labels)
print(f"labels.shape={labels.shape}, dtype={labels.dtype}")
print(f"unique labels: {unique_labels}")
print(f"voxel counts: {{{', '.join(f'{int(l)}: {int((labels == l).sum())}' for l in unique_labels if l)}}}")

In [ ]:
areas = np.zeros((10, 200, 200), dtype=np.uint64)

# Area 1 — upper-left
areas[2:9, 10:58, 10:98] = 6


# Area2 — lower-right
areas[1:10, 96:192, 58:178] = 8

unique_labels = np.unique(labels)
print(f"labels.shape={labels.shape}, dtype={labels.dtype}")
print(f"unique labels: {unique_labels}")
print(f"voxel counts: {{{', '.join(f'{int(l)}: {int((labels == l).sum())}' for l in unique_labels if l)}}}")

In [ ]:
stackview.side_by_side(labels, areas)

## RegionAnalyzer (dataframe output)

Analyze the full 3D volume (`slice_def=()`). With `map_axes=True`, vector properties such as `cross_sectional_area` and `aspect_ratio` are expanded to plane-specific columns (`-xy`, `-xz`, `-yz`).

In [ ]:
metadata = {
    "axes": ["Z", "Y", "X"],
    "scale": (2.0, 1.0, 1.0),
}

config = RegionAnalyzerConfig(
    output_type="dataframe",
    map_axes=True,
    index_on = "object_id",
    properties=[
        "label",
        "volume",
        "centroid",
        "bbox",
        "aspect_ratio",
        "cross_sectional_area",
    ],
    iterator_config=ArrayIteratorConfig(slice_def=()),
)

l_regions = RegionAnalyzer(config).run(labels, metadata=metadata)
a_regions = RegionAnalyzer(config).run(areas, metadata=metadata)

In [ ]:
l_regions

In [ ]:
a_regions

# Filter Regions

In [ ]:
rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="volume",
            range=(100.0,np.inf)
        ),
        MinFilterConfig(
            attribute="aspect_ratio",
            minimum=0.015,
        ),
    ]
)
l_accepted, _ = RegionFilter(rfcfg).run(l_regions)
a_accepted, _ = RegionFilter(rfcfg).run(a_regions)

In [ ]:
l_accepted

In [ ]:
a_accepted

# Calculate inter-object distances

Uses PyTorch tensors. The config allows setting a `preferred_device` ("cuda", "mps", "cpu", None). This is not a guarantee. The actual device can be assigned at runtime with `device`. If the device is None, it will be auto-discovered considering the config's `preferred_device`.

In [ ]:
dccfg = DistanceCalculatorConfig(
    annotate=True, 
    output_type="torch.Tensor",
    #output_type="dataframe",
    preferred_device="cuda",
)

centroids = dataframe_to_numpy(l_accepted, attributes=["centroid"], strict=False, axes=metadata["axes"])
object_ids = dataframe_to_numpy(l_accepted, attributes=["object_id"])
dist = DistanceCalculator(dccfg).run(
    centroids, 
    centroids, 
    spacing=metadata.get("scale", None), 
    point_annotations=(object_ids, object_ids),
    device=None
)

In [ ]:
dist

# Apply a Rank Filter

`axis=0`: column-wise
`axis=1`: row-wise
`axis=None`: global

`output` options: ["masked_values", "indices", "mask", "values"]

In [ ]:
tkcfg = TopKFilterConfig(
    k=1,
    axis=1,
    largest=False,
    triangle=OFF_DIAGONAL,
    output="masked_values",
)

tk = TopKFilter(tkcfg).run(dist)
tk

# Apply a Threshold based Filter

In [ ]:
mincfg = ValueFilterConfig(
    ref_value=80.0,
    axis=0,
    operator=">",
    triangle=LOWER_ND,
    output="masked_values",
)

mint = ValueFilter(mincfg).run(dist)
mint

# Aggregate

In [ ]:
macfg = MatrixAggregatorConfig(
    operation="count",
    axis=1,
)

counts = MatrixAggregator(macfg).run(mint)
counts

## Overlap calculation for coincidence detection

`OverlapCalculator` composes a **builder**, **area** calculator, **intersection** calculator, and one or more **metrics** (IoU, IoS, Dice). Pick a preset config for the input representation:

| Preset | Inputs to `.run(a, b, ...)` |
|--------|-----------------------------|
| `LabelOverlapCalculatorConfig` | 2D/3D **integer label volumes** + `region_map` |
| `BoxOverlapCalculatorConfig` | `(N, 6)` box arrays or `region_map` with bboxes only |
| `MaskOverlapCalculatorConfig` | `(N, *spatial)` boolean mask stacks |

On the **label path**, overlap is computed directly from label volumes (`label_areas`, `label_intersection_linear` / `label_intersection_sparse`). No full mask stacks are built, so large volumes (e.g. hundreds of objects on `512×512×Z`) stay memory-efficient.

### Region maps (`object_id` vs `label_id`)

After `RegionAnalyzer` and optional `RegionFilter`, build a map per channel from the accepted region table — **not** the table itself as overlap input:

```python
l_rm = region_map_from_dataframe(l_accepted.reset_index())
a_rm = region_map_from_dataframe(a_accepted.reset_index())
```

Each map entry is keyed by globally unique **`object_id`**. The value holds:

- **`label_id`** — integer label in that channel's label volume (required for the label preset)
- **`bbox`** — optional axis-aligned box (discovered from the volume when omitted)

Row/column order in the overlap matrices follows map key order (dataframe row order). With `annotate=True`, DataFrame labels default to `object_id`. Pass `annotations=(row_labels, col_labels)` to `run()` to override display names (lengths must match region counts).

### Physical spacing

Pass voxel spacing aligned with array axes, e.g. `spacing=metadata["scale"]` from `RegionAnalyzer`. Signed components encode axis direction only; magnitudes are used to scale areas and intersections. IoU / IoS / Dice ratios are unchanged under uniform scaling.

### Return value (`OverlapResult`)

`.run()` always returns an `OverlapResult` — same type for piping into downstream tasks.

| Field | Default | Notes |
|-------|---------|-------|
| `metrics` | populated | dict of raw arrays, e.g. `result.metrics["iou"]` |
| `area_a`, `area_b`, `intersection`, `union` | `None` | set `return_components=True` on the config to include |
| `object_ids_a`, `object_ids_b` | when `region_map` passed | mirror map key order |
| `annotations` | when `region_map` + `annotate=True` | used by `calc.format(result)` |

Use `calc.format(result)` (or `calc.format(result, "iou")`) to apply `output_type` and `annotate` from the config — e.g. annotated DataFrames.

### Label intersection mode

On `LabelOverlapCalculatorConfig`, set `intersection_calculator=LabelIntersectionCalculatorConfig(mode=...)`:

- **`"auto"`** (default) — pick linear (histogram) or sparse (bbox crops) from volume size and bbox overlap
- **`"linear"`** — fast when objects are dense or moderately overlapping
- **`"sparse"`** — fast when bboxes are small and mostly disjoint

### Metrics

- **IoU** — intersection / union
- **IoS** — intersection / min(area); 1.0 means one object fully contains the other
- **Dice** — 2×intersection / (area_a + area_b)


In [ ]:
olcfg = LabelOverlapCalculatorConfig(
    annotate=True,
    output_type="dataframe",
    return_components=False,  # True to also return area_a, area_b, intersection, union
    intersection_calculator=LabelIntersectionCalculatorConfig(mode="auto"),
    metrics_calculators=metrics_calculator_configs(("iou", "ios", "dice")),
)

# Region maps from filtered tables — not the DataFrames themselves.
# reset_index() exposes object_id as a column for region_map_from_dataframe.
l_rm = region_map_from_dataframe(l_accepted.reset_index())
a_rm = region_map_from_dataframe(a_accepted.reset_index())
# custom_row_labels = [f"row {i+1}" for i in range(5)]
# custom_col_labels = ["a", "b"]

calc = OverlapCalculator(olcfg)

# Pass label volumes (not region property tables) as a and b.
overlaps = calc.run(
    labels,
    areas,
    region_map=(l_rm, a_rm),
    spacing=metadata.get("scale"),
    # annotations=(custom_row_labels, custom_col_labels),  # optional display override
)

## Raw Metrics

In [ ]:
overlaps.metrics["ios"]  # 1.0: one object fully contains the other; 0.0: no overlap


In [ ]:
overlaps.metrics["iou"]

In [ ]:
overlaps.metrics["dice"]

## Annotated DataFrames (raw arrays are on overlaps.metrics)

In [ ]:
# get iou as dataframe with object annotations
iou_df = calc.format(overlaps, "iou")
iou_df